# Universe — step 2 of 8

**In plain words:** the eligible list, rebuilt for each date rather than for today.

**It produces** `Universe/Security_Master.csv` and `Universe/Data_Issues.csv`.

**It prevents** survivorship bias — testing on the winners that happened to survive.

> **Runs after `Data/curator.py`**, because it profiles the files that were downloaded, and
> **before `Data/refinery.py`**, because that stage joins the security master this one writes.

**The universe is decided in `Universe/` and nowhere else** — the seed says which securities, this
notebook what each one is and from when. Nothing downstream second-guesses either.

```
Universe/Investable_Universe.csv   the seed, committed  ->  edit this to change the universe
        |
        +--> Data/curator.py       downloads one file per identifier in it
        |
        +--> this notebook         Security_Master.csv, Data_Issues.csv
```

**This notebook is empty by design.** Each section below says what is expected in it. Write the
cells.

## 0 · Setup

Paths and the provider key, read from `Config/.env`. Nothing here touches the network, and no
value from that file is ever printed.

## The seed

`Universe/Investable_Universe.csv` is the only thing that decides what this repository is about.
Replace its rows with equities, ETFs, FX crosses, crypto pairs or futures and every stage below
still runs — nothing downstream names an asset class.

**One column is required: `main_identifier`**, the name the Data Curator asks a provider for.
Every other column is yours. Most strategies want at least a readable name and whatever they group
securities by; add those columns here and they flow through the whole pipeline.

## 1 · The seed, checked

Two things to establish before anything downstream trusts this file.

1. **Every identifier is unique**, and so is every other identity column you carry. A repeated
   identity under two rows is usually a renamed security, and the two legs have to be stitched
   into one position or the book holds it twice. A point-in-time universe is supposed to contain
   these; what it must not do is hide them.
2. **The grouping is complete.** Whatever column the strategy compares things by, a missing value
   in it is a security that silently drops out of every group-level view.

## 2 · The security master

The seed gives identity. Everything else comes from the provider: the official name, what kind of
instrument it is, where it trades, in what currency, and when it started.

**Cache the provider's raw payload, then shape the master from the cache.** Re-running then costs
nothing, changing the column mapping never triggers a refetch, and the untouched payload stays
available for fields this notebook does not yet use. Adding a provider is one fetch function and
one normaliser.

**The seed wins.** Its identity columns define the universe, so a provider value never overwrites
one — it is compared instead, and a disagreement is reported. A provider that now points a symbol
at a different security is a recycled identifier, and joining on it across the whole history would
silently mix two companies.

> **What the master carries is what a security is *today*.** The provider keeps no history, so on
> a universe whose members get reclassified, every period before the move is attributed wrongly and
> nothing raises an error. That is why the refinery prefixes every joined column `current_`.

## 3 · Composition, and how long each member has existed

Two facts decide what a backtest can honestly claim. **What the universe is made of** sets what
diversification is even available; **when each member started trading** sets the window, because a
universe is only complete from the inception of its youngest member.

## 4 · The data-issues register

The seed says what *should* exist. This section reads what *does*, and writes
`Universe/Data_Issues.csv`. Eight checks, each of which has cost somebody real time somewhere:

| Check | The failure it catches |
| --- | --- |
| **Missing file** | an identifier the provider does not carry, which becomes a silent hole in the panel |
| **Schema drift** | a folder holding two column sets, which makes every downstream read conditional |
| **No usable signal** | a history shorter than the strategy's longest warm-up, so the name can never be selected |
| **Unusable values** | zero or negative prices, which break every return calculation downstream |
| **Impossible daily move** | an adjusted price that multiplies by more than six in a day: an unadjusted corporate action or a bad print, not a return |
| **Late start** | a series that begins after the panel does, so the cross-section is smaller before that date |
| **Status disagreement** | a series that ends early on a name the provider still calls active: a data gap, not a delisting |
| **Early end** | a series that stops early — delisted or halted, and a held position must be exited on its last priced day |

Two checks the `universe-point-in-time` skill lists are not written here: **internal gaps**, and
**identity conflict**, because the provider cache is keyed by the symbol the provider returns, so
the comparison in section 2 cannot disagree, and this seed carries no second identity column to
check it against.

## 5 · When the universe is actually usable

A file that starts in 2010 gives no signal in 2010. Every feature has a warm-up, and a five-year
one moves the honest start of a backtest by five years.

**The date that matters is the first day on which every security can be both priced and
signalled.** Before it the strategy is choosing from a smaller menu than it appears to be, and a
backtest that starts earlier is quietly comparing books drawn from different universes. Nothing
else in the pipeline says so, which is why it is answered here.

## 6 · Handoff

| Output | Consumed by |
| --- | --- |
| `Universe/Security_Master.csv` | `Data/refinery.py`, which joins its columns onto the panel |
| `Universe/Data_Issues.csv` | the caveats section of every `FINDINGS_N.md` |

## Open items to carry forward

| # | Item | Why it matters |
| --- | --- | --- |
| 1 | **Classification is a snapshot, not a history.** | A security reclassified mid-window is misattributed before its move. The `current_*` prefix marks exactly this. |
| 2 | **The last day of a delisted name is unaudited.** Nothing here checks whether a truncated series ends on a real final price or on a provider gap. | On a universe that retains delisted names, the missing returns are disproportionately the bad ones. |
| 3 | **Inception dates come from the provider, not from the price file.** | The price file is what the backtest actually trades; the master is what a reader believes. Where they disagree, say so. |